# CLIP ZERO SHOT PIPELINE
In this notebook we test our first methodology. We started from CLIP zero shot to get our first results and get a baseline for the results. Since we do not have textual labels for the query images we expect this pipeline to perform very poorly

In [2]:
# Paths definition

import os

DATASET_PATH = '/kaggle/input/competitions/aml-group-project/release'
TRAIN_CSV_PATH = '/kaggle/input/competitions/aml-group-project/release/train.csv'
IMGS_PATH = '/kaggle/input/competitions/aml-group-project/release/images'
TEST_CSV_PATH = '/kaggle/input/competitions/aml-group-project/release/test_episodes_release.csv'
SUBMISSION_CSV_PATH = '/kaggle/input/competitions/aml-group-project/release/sample_submission.csv'

To avoid running out of submissions on kaggle, we can test this metholodgy by exploiting hypothetical episodes created from the train set (were labels are available for all images)
This cell is a function that generate a fictitious episode

In [3]:
import pandas as pd
import numpy as np

def generate_episode_from_train(train_df, way=5, shot=5, query_per_class=5):
    
    """
    Generate a fictitious episode starting from the train dataframe.
    Returns back the support and query subsets with visible labels.
    This functions is usefull to have some local tests before submitting on kaggle.
    """
    
    # 1. Extract the available classes of the train set and choose 5 random classes
    available_classes = train_df['label'].unique()
    selected_classes = np.random.choice(available_classes, size=way, replace=False)
    
    support_rows = []
    query_rows   = []
    
    # 2. For each class in selected_classes, we extract the support and query images
    for idx, cls in enumerate(selected_classes):

        # 3. Extract only the rows of the df corresponding to the class selected in the for loop
        class_subset = train_df[train_df['label'] == cls]
        
        # 4. Extract 10 random rows of this subset of the original df (5 for the support and 5 for the query)
        sampled_rows = class_subset.sample(n= shot + query_per_class, replace=False)
        
        # 5. Select 5 rows for the support and 5 for the query
        support_part = sampled_rows.iloc[:shot].copy()
        query_part = sampled_rows.iloc[shot:].copy()
        
        # 6. Assign labels to each class (from 0 to 4)
        support_part['local_label'] = idx
        query_part['local_label'] = idx
        
        support_rows.append(support_part)
        query_rows.append(query_part)
        
    # Merge everything to get the final structure for the hyphotetical episode
    episode_support_df = pd.concat(support_rows).sample(frac=1).reset_index(drop=True)
    episode_query_df = pd.concat(query_rows).sample(frac=1).reset_index(drop=True)
    
    return episode_support_df, episode_query_df

In [ ]:
 pip install ftfy regex tqdm git+https://github.com/openai/CLIP.git

In [19]:
import torch
import numpy as np
import pandas as pd
from PIL import Image
import os
from tqdm import tqdm
import clip

# 1. Setting the constants for the zero shot approach
WAY = 5
SHOT = 5
QUERY_PER_CLASS = 5
NUM_EPISODES = 100  # So we can cover the whole train dataset

# 2. If the GPU is available set it as the main device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device used for the computation: {device}")

# 3. Now we can load clip (model + transforms + select inference mode)
model, preprocess = clip.load("ViT-B/32", device=device)
model.eval()

df_train = pd.read_csv(TRAIN_CSV_PATH)
all_episode_accuracies = [] # Place holder to save the accuracy of each episode

print(f"Starting the accuracy computation on {NUM_EPISODES} random episodes")

for ep in tqdm(range(NUM_EPISODES)):

    # 4. At each iteration we generate a new random episode
    support_df, query_df = generate_episode_from_train(df_train, way=WAY, shot=SHOT, query_per_class=QUERY_PER_CLASS)

    # 5. Textual prompts for this episode: THIS IS THE DRAWBACK OF HAVING NUMERICAL LABELS
    text_prompts = [f"a photo of a class {i}" for i in range(WAY)]
    
    # 6. clip.tokenize convert the text into token ready for the Text Encoder
    text_tokens = clip.tokenize(text_prompts).to(device)
    
    with torch.no_grad():
        # 7. Extract and normalize textual features
        text_features = model.encode_text(text_tokens)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)



    # 8. Starting the processing and inference on query images
    query_embeddings = []
    ground_truth_labels = query_df['local_label'].values # Ground truths (0, 1, 2, 3 o 4)

    for _, row in query_df.iterrows():

        # 9. Load each query img and preprocess it so that it is ready for the Image Encoder
        img_path = os.path.join(IMGS_PATH, row['filename'])
        img = Image.open(img_path).convert('RGB')
        img_input = preprocess(img).unsqueeze(0).to(device)

        with torch.no_grad():

            # 10. Extract and normalize the embeddings for the images
            img_feature = model.encode_image(img_input)
            img_feature = img_feature / img_feature.norm(dim=-1, keepdim=True)
            query_embeddings.append(img_feature)

    query_embeddings = torch.cat(query_embeddings, dim=0)


    # 11. Computation of Cosine Similarity and predictions
    similarity = (query_embeddings @ text_features.T)
    predictions = similarity.argmax(dim=-1).cpu().numpy()

    ep_accuracy = np.mean(predictions == ground_truth_labels)
    all_episode_accuracies.append(ep_accuracy)


# 12. Compute final metrics and statistics
mean_accuracy = np.mean(all_episode_accuracies) # Average accuracy between all episodes
std_accuracy = np.std(all_episode_accuracies)   # Standard Deviation 
confidence_interval = 1.96 * (std_accuracy / np.sqrt(NUM_EPISODES)) # z distribution

print("\n==================================================")
print("   FINAL RESULTS (OPENAI CLIP ZERO-SHOT) ")
print("==================================================")
print(f"Confidence Interval (95%): ±{confidence_interval * 100:.2f}%")
print(f"Average accuracy on {NUM_EPISODES} episodi: {mean_accuracy * 100:.2f}%")
print(f"Standard deviation:            {std_accuracy * 100:.2f}%")
print("==================================================")

Device used for the computation: cuda


100%|████████████████████████████████████████| 338M/338M [00:02<00:00, 126MiB/s]


Starting the accuracy computation on 100 random episodes


100%|██████████| 100/100 [00:49<00:00,  2.02it/s]


   FINAL RESULTS (OPENAI CLIP ZERO-SHOT) 
Intervallo di Confidenza (95%): ±1.73%
Average accuracy on 100 episodi: 20.36%
Standard deviation:            8.80%


The model is randomnly guessing because it reaches a 20% accuracy! The thin confidence interval prove that this is not due to the case but it is statistically significant
In the next attempt, we will try to exploit more the support set